# 07 Retrieval Baseline (Document-Known Dense Retrieval)

## Σκοπός
Σε αυτό το notebook:

- φορτώνουμε το FinanceBench working dataset
- φορτώνουμε τα chunks και τα embeddings
- για κάθε query χρησιμοποιούμε το γνωστό `doc_name`
- κάνουμε dense retrieval μόνο μέσα στο σωστό document
- αποθηκεύουμε τα top-k retrieval results

Στόχος είναι να έχουμε ένα καθαρό document-known dense retrieval baseline.

In [98]:
# Uncomment if needed
# !pip install -q sentence-transformers faiss-cpu pyarrow tqdm

In [99]:
from pathlib import Path
import json
import warnings
import pickle
import re

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import faiss
import torch
from sentence_transformers import SentenceTransformer

In [100]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 180)

IN_KAGGLE = Path("/kaggle/working").exists()
print("IN_KAGGLE:", IN_KAGGLE)
print("CUDA available:", torch.cuda.is_available())

IN_KAGGLE: False
CUDA available: False


In [101]:
EMBEDDING_MODEL = "BAAI/bge-m3"
TOP_K = 10

USE_QUERY_LIMIT = False
QUERY_LIMIT = 50

retrieval_config = {
    "retrieval_type": "dense",
    "embedding_model": EMBEDDING_MODEL,
    "top_k": TOP_K,
    "document_known": False,
    "query_expansion": True
}

retrieval_config

{'retrieval_type': 'dense',
 'embedding_model': 'BAAI/bge-m3',
 'top_k': 10,
 'document_known': False,
 'query_expansion': True}

In [102]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

CHUNKS_DIR = PROCESSED_DIR / "chunks"
EMBEDDINGS_DIR = PROCESSED_DIR / "embeddings"
RETRIEVAL_DIR = PROCESSED_DIR / "retrieval_results"

WORKING_DATASET_CSV_PATH = INTERIM_DIR / "financebench_open_source_working.csv"
WORKING_DATASET_PARQUET_PATH = INTERIM_DIR / "financebench_open_source_working.parquet"

CHUNKS_CSV_PATH = CHUNKS_DIR / "financebench_chunks.csv"
CHUNKS_PARQUET_PATH = CHUNKS_DIR / "financebench_chunks.parquet"

EMBEDDINGS_MATRIX_PATH = EMBEDDINGS_DIR / "chunk_embeddings.npy"
EMBEDDINGS_METADATA_CSV_PATH = EMBEDDINGS_DIR / "chunk_embeddings_metadata.csv"
EMBEDDINGS_METADATA_PARQUET_PATH = EMBEDDINGS_DIR / "chunk_embeddings_metadata.parquet"

RETRIEVAL_RESULTS_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_dense.csv"
RETRIEVAL_RESULTS_PARQUET_PATH = RETRIEVAL_DIR / "retrieval_results_dense.parquet"
RETRIEVAL_MANIFEST_PATH = RETRIEVAL_DIR / "retrieval_manifest_dense.csv"
RETRIEVAL_STATS_PATH = RETRIEVAL_DIR / "retrieval_stats_dense.json"

RETRIEVAL_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RETRIEVAL_DIR:", RETRIEVAL_DIR)

BASE_DIR: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag
RETRIEVAL_DIR: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\retrieval_results


In [103]:
if WORKING_DATASET_PARQUET_PATH.exists():
    query_df = pd.read_parquet(WORKING_DATASET_PARQUET_PATH)
elif WORKING_DATASET_CSV_PATH.exists():
    query_df = pd.read_csv(WORKING_DATASET_CSV_PATH)
else:
    raise FileNotFoundError("Working dataset not found.")

print("query_df shape before filtering:", query_df.shape)
query_df.head(2)

query_df shape before filtering: (150, 22)


,row_id,financebench_id,question,answer,company,doc_name,doc_type,doc_period,pdf_filename,pdf_path,question_type,question_reasoning,justification,evidence,domain_question_num,dataset_subset_label,company_doc,gics_sector,doc_link,normalized_doc_name,pdf_stem,normalized_pdf_stem
0,0,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,$1577.00,3M,3M_2018_10K,10k,2018,3M_2018_10K.pdf,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\3M_2018_10K.pdf,metrics-generated,Information extraction,"The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).","[{'doc_name': '3M_2018_10K', 'evidence_page_num': 59, 'evidence_text': 'Table of Contents 3M Company and Subsidiaries Consolidated Statement of Cash Flow s Years ended December 31 (Millions) ...",None,OPEN_SOURCE,3M,Industrials,https://investors.3m.com/financials/sec-filings/content/0001558370-19-000470/0001558370-19-000470.pdf,3m 2018 10k,3M_2018_10K,3m 2018 10k
1,1,financebench_id_04672,Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer ...,$8.70,3M,3M_2018_10K,10k,2018,3M_2018_10K.pdf,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\3M_2018_10K.pdf,metrics-generated,Information extraction,"The metric ppne, net was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Property, plant and equipment â net.","[{'doc_name': '3M_2018_10K', 'evidence_page_num': 57, 'evidence_text': 'Table of Contents 3M Company and Subsidiaries Consolidated Balance Shee t At December 31 December 31, December 31, ...",None,OPEN_SOURCE,3M,Industrials,https://investors.3m.com/financials/sec-filings/content/0001558370-19-000470/0001558370-19-000470.pdf,3m 2018 10k,3M_2018_10K,3m 2018 10k


In [104]:
required_cols = ["question", "financebench_id"]

missing_cols = [c for c in required_cols if c not in query_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in query_df: {missing_cols}")

query_df = query_df.copy()

# create question_clean if missing
if "question_clean" not in query_df.columns:
    query_df["question_clean"] = query_df["question"].astype(str)

query_df["question_clean"] = (
    query_df["question_clean"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

query_df = query_df[query_df["question_clean"].notna()].copy().reset_index(drop=True)

if USE_QUERY_LIMIT:
    query_df = query_df.head(QUERY_LIMIT).copy().reset_index(drop=True)

print("query_df shape after filtering:", query_df.shape)
print(query_df.columns.tolist())
query_df[["financebench_id", "question", "question_clean"]].head(3)

query_df shape after filtering: (150, 23)
['row_id', 'financebench_id', 'question', 'answer', 'company', 'doc_name', 'doc_type', 'doc_period', 'pdf_filename', 'pdf_path', 'question_type', 'question_reasoning', 'justification', 'evidence', 'domain_question_num', 'dataset_subset_label', 'company_doc', 'gics_sector', 'doc_link', 'normalized_doc_name', 'pdf_stem', 'normalized_pdf_stem', 'question_clean']


,financebench_id,question,question_clean
0,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
1,financebench_id_04672,Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer ...,Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer ...
2,financebench_id_00499,Is 3M a capital-intensive business based on FY2022 data?,Is 3M a capital-intensive business based on FY2022 data?


In [105]:
if CHUNKS_PARQUET_PATH.exists():
    chunks_df = pd.read_parquet(CHUNKS_PARQUET_PATH)
elif CHUNKS_CSV_PATH.exists():
    chunks_df = pd.read_csv(CHUNKS_CSV_PATH)
else:
    raise FileNotFoundError("Chunks file not found.")

print("chunks_df shape:", chunks_df.shape)
print(chunks_df.columns.tolist())
chunks_df.head(2)

chunks_df shape: (37037, 16)
['chunk_id', 'doc_id', 'doc_name', 'markdown_filename', 'source_path', 'company', 'doc_type', 'doc_period', 'chunk_index', 'chunk_text', 'char_count', 'token_estimate', 'starts_with_heading', 'contains_table_pipe', 'is_table_chunk', 'has_table_header']


,chunk_id,doc_id,doc_name,markdown_filename,source_path,company,doc_type,doc_period,chunk_index,chunk_text,char_count,token_estimate,starts_with_heading,contains_table_pipe,is_table_chunk,has_table_header
0,3M_2018_10K_chunk_0000,3M_2018_10K,3M_2018_10K,3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2018_10K.md,None,None,None,0,"## UNITED STATES SECURITIES AND EXCHANGE COMMISSION\n\nWashington, D.C. 20549\n\n## FORM 10-K\n\n☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fisc...",540,135,True,False,False,False
1,3M_2018_10K_chunk_0001,3M_2018_10K,3M_2018_10K,3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2018_10K.md,None,None,None,1,"## Title of each class\n\non which registered\n\nCommon Stock, Par Value $.01 Per Share\n\n1.500% Notes due 2026\n\nFloating Rate Notes due 2020\n\n0.375% Notes due 2022\n\n0.950% Notes due 2023\n...",1271,317,True,False,False,False


In [106]:
from pathlib import Path

print("BASE_DIR:", BASE_DIR)
print("CHUNKS_DIR exists:", CHUNKS_DIR.exists())
print("CHUNKS_DIR:", CHUNKS_DIR)

if CHUNKS_DIR.exists():
    print("Files inside CHUNKS_DIR:")
    for p in sorted(CHUNKS_DIR.iterdir()):
        print("-", p.name)

BASE_DIR: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag
CHUNKS_DIR exists: True
CHUNKS_DIR: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\chunks
Files inside CHUNKS_DIR:
- chunking_manifest.csv
- chunking_stats.json
- financebench_chunks.csv
- financebench_chunks.parquet
- financebench_chunks.zip


In [107]:
if EMBEDDINGS_METADATA_PARQUET_PATH.exists():
    embeddings_metadata_df = pd.read_parquet(EMBEDDINGS_METADATA_PARQUET_PATH)
elif EMBEDDINGS_METADATA_CSV_PATH.exists():
    embeddings_metadata_df = pd.read_csv(EMBEDDINGS_METADATA_CSV_PATH)
else:
    raise FileNotFoundError("Embeddings metadata file not found.")

if not EMBEDDINGS_MATRIX_PATH.exists():
    raise FileNotFoundError("Embeddings matrix .npy file not found.")

embeddings_matrix = np.load(EMBEDDINGS_MATRIX_PATH)

print("embeddings_metadata_df shape:", embeddings_metadata_df.shape)
print("embeddings_matrix shape:", embeddings_matrix.shape)

assert len(embeddings_metadata_df) == len(embeddings_matrix), "Metadata and embeddings row count mismatch"

print(embeddings_metadata_df.columns.tolist())
embeddings_metadata_df.head(2)

embeddings_metadata_df shape: (37037, 11)
embeddings_matrix shape: (37037, 1024)
['chunk_id', 'doc_id', 'chunk_index', 'company', 'doc_type', 'doc_period', 'char_count', 'token_estimate', 'embedding_dim', 'text_len', 'text_preview']


,chunk_id,doc_id,chunk_index,company,doc_type,doc_period,char_count,token_estimate,embedding_dim,text_len,text_preview
0,3M_2018_10K_chunk_0000,3M_2018_10K,0,NaN,NaN,NaN,540,135,1024,540,"## UNITED STATES SECURITIES AND EXCHANGE COMMISSION\n\nWashington, D.C. 20549\n\n## FORM 10-K\n\n☒ ANNUAL REPORT PURSUANT TO S"
1,3M_2018_10K_chunk_0001,3M_2018_10K,1,NaN,NaN,NaN,1271,317,1024,1271,"## Title of each class\n\non which registered\n\nCommon Stock, Par Value $.01 Per Share\n\n1.500% Notes due 2026\n\nFloating Rat"


In [108]:
retrieval_corpus_df = chunks_df.copy().reset_index(drop=True)

assert len(retrieval_corpus_df) == len(embeddings_matrix), \
    "Chunks dataframe and embeddings matrix row count mismatch"

print("retrieval_corpus_df shape:", retrieval_corpus_df.shape)
print(retrieval_corpus_df.columns.tolist())
retrieval_corpus_df.head(2)

retrieval_corpus_df shape: (37037, 16)
['chunk_id', 'doc_id', 'doc_name', 'markdown_filename', 'source_path', 'company', 'doc_type', 'doc_period', 'chunk_index', 'chunk_text', 'char_count', 'token_estimate', 'starts_with_heading', 'contains_table_pipe', 'is_table_chunk', 'has_table_header']


,chunk_id,doc_id,doc_name,markdown_filename,source_path,company,doc_type,doc_period,chunk_index,chunk_text,char_count,token_estimate,starts_with_heading,contains_table_pipe,is_table_chunk,has_table_header
0,3M_2018_10K_chunk_0000,3M_2018_10K,3M_2018_10K,3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2018_10K.md,None,None,None,0,"## UNITED STATES SECURITIES AND EXCHANGE COMMISSION\n\nWashington, D.C. 20549\n\n## FORM 10-K\n\n☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fisc...",540,135,True,False,False,False
1,3M_2018_10K_chunk_0001,3M_2018_10K,3M_2018_10K,3M_2018_10K.md,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\cleaned\3M_2018_10K.md,None,None,None,1,"## Title of each class\n\non which registered\n\nCommon Stock, Par Value $.01 Per Share\n\n1.500% Notes due 2026\n\nFloating Rate Notes due 2020\n\n0.375% Notes due 2022\n\n0.950% Notes due 2023\n...",1271,317,True,False,False,False


In [109]:
required_chunk_cols = ["chunk_id", "doc_id", "chunk_text"]

missing_chunk_cols = [c for c in required_chunk_cols if c not in retrieval_corpus_df.columns]
if missing_chunk_cols:
    raise ValueError(f"Missing required columns in retrieval_corpus_df: {missing_chunk_cols}")

print("Retrieval corpus columns OK.")

Retrieval corpus columns OK.


In [110]:
FINANCE_ALIAS_MAP = {
    "ppe": [
        "property plant equipment",
        "property plant and equipment",
        "property plant and equipment net",
        "pp&e",
        "balance sheet"
    ],
    "net ppe": [
        "property plant and equipment net",
        "net property plant equipment",
        "property plant equipment net",
        "balance sheet"
    ],
    "capex": [
        "capital expenditure",
        "capital expenditures",
        "purchases of property plant and equipment",
        "purchases of property, plant and equipment",
        "cash flow statement"
    ],
    "cogs": [
        "cost of goods sold",
        "cost of sales",
        "cost of products sold",
        "income statement"
    ],
    "ebitda": [
        "earnings before interest taxes depreciation and amortization",
        "non-gaap operating performance"
    ],
    "opex": [
        "operating expenses",
        "selling general and administrative",
        "sg&a"
    ],
    "gross margin": [
        "gross profit margin",
        "gross profit",
        "income statement"
    ],
    "operating cash flow": [
        "net cash provided by operating activities",
        "cash flow statement"
    ],
    "free cash flow": [
        "free cash flow",
        "net cash provided by operating activities",
        "capital expenditures"
    ],
    "working capital": [
        "current assets",
        "current liabilities",
        "balance sheet"
    ],
    "payout ratio": [
        "dividends",
        "net income",
        "dividend payout ratio"
    ],
    "retention ratio": [
        "retained earnings ratio",
        "dividend payout ratio",
        "dividends",
        "net income"
    ],
}

In [111]:
def normalize_query_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def expand_finance_query(query: str) -> str:
    q = normalize_query_text(query)
    expansions = []

    ordered_aliases = sorted(FINANCE_ALIAS_MAP.keys(), key=len, reverse=True)

    for alias in ordered_aliases:
        if alias in q:
            expansions.extend(FINANCE_ALIAS_MAP[alias])

    if "balance sheet" in q:
        expansions.extend(["balance sheet", "assets liabilities equity"])
    if "cash flow" in q:
        expansions.extend(["cash flow statement", "operating activities investing activities financing activities"])
    if "income statement" in q:
        expansions.extend(["income statement", "net sales operating income net income"])
    if "year end" in q or "year-end" in q:
        expansions.extend(["at december 31", "balance sheet"])

    seen = set()
    cleaned_expansions = []

    for item in expansions:
        item = item.strip().lower()
        if item and item not in seen:
            seen.add(item)
            cleaned_expansions.append(item)

    if cleaned_expansions:
        return q + " " + " ".join(cleaned_expansions)

    return q

In [112]:
query_df["question_clean"] = query_df["question_clean"].astype(str)
query_df["expanded_question"] = query_df["question_clean"].apply(expand_finance_query)

query_df[["financebench_id", "question_clean", "expanded_question"]].head(10)

,financebench_id,question_clean,expanded_question
0,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,what is the fy2018 capital expenditure amount (in usd millions) for 3m? give a response to the question by relying on the details shown in the cash flow statement. cash flow statement operating ac...
1,financebench_id_04672,Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer ...,assume that you are a public equities analyst. answer the following question by primarily using information that is shown in the balance sheet: what is the year end fy2018 net ppne for 3m? answer ...
2,financebench_id_00499,Is 3M a capital-intensive business based on FY2022 data?,is 3m a capital-intensive business based on fy2022 data?
3,financebench_id_01226,"What drove operating margin change as of FY2022 for 3M? If operating margin is not a useful metric for a company like this, then please state that and explain why.","what drove operating margin change as of fy2022 for 3m? if operating margin is not a useful metric for a company like this, then please state that and explain why."
4,financebench_id_01865,"If we exclude the impact of M&A, which segment has dragged down 3M's overall growth in 2022?","if we exclude the impact of m&a, which segment has dragged down 3m's overall growth in 2022?"
5,financebench_id_00807,"Does 3M have a reasonably healthy liquidity profile based on its quick ratio for Q2 of FY2023? If the quick ratio is not relevant to measure liquidity, please state that and explain why.","does 3m have a reasonably healthy liquidity profile based on its quick ratio for q2 of fy2023? if the quick ratio is not relevant to measure liquidity, please state that and explain why."
6,financebench_id_00941,Which debt securities are registered to trade on a national securities exchange under 3M's name as of Q2 of 2023?,which debt securities are registered to trade on a national securities exchange under 3m's name as of q2 of 2023?
7,financebench_id_01858,Does 3M maintain a stable trend of dividend distribution?,does 3m maintain a stable trend of dividend distribution?
8,financebench_id_02987,What is the FY2019 fixed asset turnover ratio for Activision Blizzard? Fixed asset turnover ratio is defined as: FY2019 revenue / (average PP&E between FY2018 and FY2019). Round your answer to two...,what is the fy2019 fixed asset turnover ratio for activision blizzard? fixed asset turnover ratio is defined as: fy2019 revenue / (average pp&e between fy2018 and fy2019). round your answer to two...
9,financebench_id_07966,What is the FY2017 - FY2019 3 year average of capex as a % of revenue for Activision Blizzard? Answer in units of percents and round to one decimal place. Calculate (or extract) the answer from th...,what is the fy2017 - fy2019 3 year average of capex as a % of revenue for activision blizzard? answer in units of percents and round to one decimal place. calculate (or extract) the answer from th...


In [113]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print("Loaded embedding model:", EMBEDDING_MODEL)

Loaded embedding model: BAAI/bge-m3


In [114]:
def embed_queries(texts):
    vectors = embedding_model.encode(
        texts,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    return np.asarray(vectors, dtype="float32")

In [115]:
def build_global_index(embeddings: np.ndarray):
    vectors = embeddings.astype("float32").copy()
    faiss.normalize_L2(vectors)

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index


def search_global(query_vector: np.ndarray, index, top_k: int):
    scores, indices = index.search(query_vector, k=min(top_k, index.ntotal))
    return scores[0], indices[0]

In [116]:
global_index = build_global_index(embeddings_matrix)
print("Global FAISS index size:", global_index.ntotal)

Global FAISS index size: 37037


In [117]:
query_vectors = embed_queries(query_df["expanded_question"].tolist())
print("query_vectors shape:", query_vectors.shape)

Batches: 100%|██████████| 5/5 [00:16<00:00,  3.25s/it]

query_vectors shape: (150, 1024)


In [118]:
retrieval_records = []
manifest_records = []

for query_idx, (_, row) in enumerate(
    tqdm(query_df.iterrows(), total=len(query_df), desc="Running dense retrieval (document-unknown)")
):
    financebench_id = row.get("financebench_id")
    expected_doc_name = row.get("doc_name")
    question = row["question"]
    question_clean = row["question_clean"]
    expanded_question = row["expanded_question"]

    manifest_record = {
        "query_row": query_idx,
        "financebench_id": financebench_id,
        "question": question,
        "expected_doc_name": expected_doc_name,
        "status": None,
        "error_message": None,
        "n_results": 0
    }

    try:
        qvec = query_vectors[query_idx].reshape(1, -1)
        scores, global_indices = search_global(
            query_vector=qvec,
            index=global_index,
            top_k=TOP_K
        )

        for rank, (score, global_idx) in enumerate(zip(scores, global_indices), start=1):
            matched_row = retrieval_corpus_df.iloc[int(global_idx)]

            retrieval_records.append({
            "query_row": query_idx,
             "financebench_id": financebench_id,
              "question": question,
             "question_clean": question_clean,
              "expanded_question": expanded_question,
              "expected_doc_name": expected_doc_name,
              "expected_company": row.get("company"),
              "retrieved_rank": rank,
             "retrieval_score": float(score),
               "embedding_row_idx": int(global_idx),
              "chunk_id": matched_row["chunk_id"],
              "retrieved_doc_id": matched_row["doc_id"],
              "chunk_index": matched_row.get("chunk_index"),
               "chunk_text": matched_row["chunk_text"],
              "char_count": matched_row.get("char_count"),
               "token_estimate": matched_row.get("token_estimate"),
                })

        manifest_record["status"] = "success"
        manifest_record["n_results"] = len(global_indices)

    except Exception as e:
        manifest_record["status"] = "error"
        manifest_record["error_message"] = str(e)

    manifest_records.append(manifest_record)

retrieval_results_df = pd.DataFrame(retrieval_records)
retrieval_manifest_df = pd.DataFrame(manifest_records)

print("retrieval_results_df shape:", retrieval_results_df.shape)
print("retrieval_manifest_df shape:", retrieval_manifest_df.shape)

Running dense retrieval (document-unknown): 100%|██████████| 150/150 [00:00<00:00, 194.79it/s]

retrieval_results_df shape: (1500, 16)
retrieval_manifest_df shape: (150, 7)


In [119]:
retrieval_results_df["doc_match"] = (
    retrieval_results_df["expected_doc_name"].fillna("").astype(str)
    == retrieval_results_df["retrieved_doc_id"].fillna("").astype(str)
)

retrieval_results_df[[
    "financebench_id",
    "retrieved_rank",
    "retrieved_doc_id",
    "expected_doc_name",
    "doc_match",
    "retrieval_score"
]].head(15)

,financebench_id,retrieved_rank,retrieved_doc_id,expected_doc_name,doc_match,retrieval_score
0,financebench_id_03029,1,WALMART_2018_10K,3M_2018_10K,False,0.698650
1,financebench_id_03029,2,3M_2018_10K,3M_2018_10K,True,0.695058
2,financebench_id_03029,3,3M_2022_10K,3M_2018_10K,False,0.691014
3,financebench_id_03029,4,3M_2023Q2_10Q,3M_2018_10K,False,0.690825
4,financebench_id_03029,5,MGMRESORTS_2018_10K,3M_2018_10K,False,0.690268
5,financebench_id_03029,6,ADOBE_2017_10K,3M_2018_10K,False,0.678058
6,financebench_id_03029,7,3M_2018_10K,3M_2018_10K,True,0.677116
7,financebench_id_03029,8,3M_2018_10K,3M_2018_10K,True,0.676206
8,financebench_id_03029,9,LOCKHEEDMARTIN_2022_10K,3M_2018_10K,False,0.673426
9,financebench_id_03029,10,AMERICANWATERWORKS_2021_10K,3M_2018_10K,False,0.666687


In [120]:
top1_df = retrieval_results_df[retrieval_results_df["retrieved_rank"] == 1].copy()

top1_doc_match_rate = float(top1_df["doc_match"].mean()) if len(top1_df) else 0.0
topk_doc_match_rate = float(
    retrieval_results_df.groupby("financebench_id")["doc_match"].max().mean()
) if len(retrieval_results_df) else 0.0

summary_df = pd.DataFrame([{
    "n_queries": int(query_df["financebench_id"].nunique()),
    "top1_doc_match_rate": top1_doc_match_rate,
    f"top{TOP_K}_doc_match_rate": topk_doc_match_rate
}])

summary_df

,n_queries,top1_doc_match_rate,top10_doc_match_rate
0,150,0.473333,0.886667


In [121]:
retrieval_results_df.to_csv(RETRIEVAL_RESULTS_CSV_PATH, index=False, encoding="utf-8")
retrieval_results_df.to_parquet(RETRIEVAL_RESULTS_PARQUET_PATH, index=False)

retrieval_manifest_df.to_csv(RETRIEVAL_MANIFEST_PATH, index=False, encoding="utf-8")
print("Saved:")
print("-", RETRIEVAL_RESULTS_CSV_PATH)
print("-", RETRIEVAL_RESULTS_PARQUET_PATH)
print("-", RETRIEVAL_MANIFEST_PATH)

Saved:
- C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\retrieval_results\retrieval_results_dense.csv
- C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\retrieval_results\retrieval_results_dense.parquet
- C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\retrieval_results\retrieval_manifest_dense.csv


In [122]:
retrieval_stats = {
    "retrieval_type": "dense",
    "document_known": False,
    "query_expansion": True,
    "embedding_model": EMBEDDING_MODEL,
    "top_k": TOP_K,
    "n_queries": int(query_df["financebench_id"].nunique()),
    "n_result_rows": int(len(retrieval_results_df)),
    "n_manifest_rows": int(len(retrieval_manifest_df)),
    "top1_doc_match_rate": top1_doc_match_rate,
    f"top{TOP_K}_doc_match_rate": topk_doc_match_rate,
    "results_csv": str(RETRIEVAL_RESULTS_CSV_PATH),
    "results_parquet": str(RETRIEVAL_RESULTS_PARQUET_PATH),
    "manifest_csv": str(RETRIEVAL_MANIFEST_PATH),
}

with open(RETRIEVAL_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(retrieval_stats, f, indent=2, ensure_ascii=False)

print("Saved stats:", RETRIEVAL_STATS_PATH)
retrieval_stats

Saved stats: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\retrieval_results\retrieval_stats_dense.json


{'retrieval_type': 'dense',
 'document_known': False,
 'query_expansion': True,
 'embedding_model': 'BAAI/bge-m3',
 'top_k': 10,
 'n_queries': 150,
 'n_result_rows': 1500,
 'n_manifest_rows': 150,
 'top1_doc_match_rate': 0.47333333333333333,
 'top10_doc_match_rate': 0.8866666666666667,
 'results_csv': 'C:\\Users\\ntheo\\Desktop\\Repositories\\Financial-RAG\\thesis-rag\\data\\processed\\retrieval_results\\retrieval_results_dense.csv',
 'results_parquet': 'C:\\Users\\ntheo\\Desktop\\Repositories\\Financial-RAG\\thesis-rag\\data\\processed\\retrieval_results\\retrieval_results_dense.parquet',
 'manifest_csv': 'C:\\Users\\ntheo\\Desktop\\Repositories\\Financial-RAG\\thesis-rag\\data\\processed\\retrieval_results\\retrieval_manifest_dense.csv'}

In [123]:
retrieval_results_df[[
    "financebench_id",
    "question",
    "expanded_question",
    "retrieved_rank",
    "retrieved_doc_id",
    "expected_doc_name",
    "doc_match",
    "retrieval_score",
    "chunk_id"
]].head(20)

,financebench_id,question,expanded_question,retrieved_rank,retrieved_doc_id,expected_doc_name,doc_match,retrieval_score,chunk_id
0,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,what is the fy2018 capital expenditure amount (in usd millions) for 3m? give a response to the question by relying on the details shown in the cash flow statement. cash flow statement operating ac...,1,WALMART_2018_10K,3M_2018_10K,False,0.698650,WALMART_2018_10K_chunk_0182
1,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,what is the fy2018 capital expenditure amount (in usd millions) for 3m? give a response to the question by relying on the details shown in the cash flow statement. cash flow statement operating ac...,2,3M_2018_10K,3M_2018_10K,True,0.695058,3M_2018_10K_chunk_0172
2,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,what is the fy2018 capital expenditure amount (in usd millions) for 3m? give a response to the question by relying on the details shown in the cash flow statement. cash flow statement operating ac...,3,3M_2022_10K,3M_2018_10K,False,0.691014,3M_2022_10K_chunk_0140
3,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,what is the fy2018 capital expenditure amount (in usd millions) for 3m? give a response to the question by relying on the details shown in the cash flow statement. cash flow statement operating ac...,4,3M_2023Q2_10Q,3M_2018_10K,False,0.690825,3M_2023Q2_10Q_chunk_0274
4,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,what is the fy2018 capital expenditure amount (in usd millions) for 3m? give a response to the question by relying on the details shown in the cash flow statement. cash flow statement operating ac...,5,MGMRESORTS_2018_10K,3M_2018_10K,False,0.690268,MGMRESORTS_2018_10K_chunk_0142
5,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,what is the fy2018 capital expenditure amount (in usd millions) for 3m? give a response to the question by relying on the details shown in the cash flow statement. cash flow statement operating ac...,6,ADOBE_2017_10K,3M_2018_10K,False,0.678058,ADOBE_2017_10K_chunk_0199
6,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,what is the fy2018 capital expenditure amount (in usd millions) for 3m? give a response to the question by relying on the details shown in the cash flow statement. cash flow statement operating ac...,7,3M_2018_10K,3M_2018_10K,True,0.677116,3M_2018_10K_chunk_0157
7,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,what is the fy2018 capital expenditure amount (in usd millions) for 3m? give a response to the question by relying on the details shown in the cash flow statement. cash flow statement operating ac...,8,3M_2018_10K,3M_2018_10K,True,0.676206,3M_2018_10K_chunk_0290
8,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,what is the fy2018 capital expendi

## Συμπέρασμα

Σε αυτό το notebook:

- εκτελέσαμε dense retrieval baseline
- χρησιμοποιήσαμε document-known retrieval
- αναζητήσαμε μόνο μέσα στο σωστό report για κάθε query
- αποθηκεύσαμε retrieval results και manifest

Το επόμενο notebook θα είναι το hybrid retrieval baseline.